# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

# import sys
# from awsglue.transforms import *
# from awsglue.utils import getResolvedOptions
# from pyspark.context import SparkContext
# from awsglue.context import GlueContext
# from awsglue.job import Job
  
# sc = SparkContext.getOrCreate()
# glueContext = GlueContext(sc)
# spark = glueContext.spark_session
# job = Job(glueContext)

In [2]:
%stop_session

Stopping session: 83569970-36cf-43c0-b7cb-54ea85026426
Stopped session.


In [4]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

The following configurations have been updated: {'--datalake-formats': 'delta', '--conf': 'spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore', '--enable-glue-datacatalog': 'true'}


In [1]:
# df = spark.range(5)
# df.write.format("delta").mode("overwrite").save("s3://data-lake-case-hotmart/_delta_smoke_test")
# spark.read.format("delta").load("s3://data-lake-case-hotmart/_delta_smoke_test").show()

Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: f26c927a-4f9a-4f01-b269-22a27d79701a
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--datalake-formats delta
--conf spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore
Waiting for session f26c927a-4f9a-4f01-b269-22a27d79701a to get into ready status...
Session f26c927a-4f9a-4f01-b269-22a27d79701a has been created.
+---+
| id|
+---+
|  4|
|  1|
|  2|
|  0|
|  3|
+---+


#ETL

In [4]:
#Setup

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import date, timedelta

BASE = "s3://data-lake-case-hotmart"

# D-1 (pode fixar uma data no UAT, ex: "2023-03-12")
RUN_DATE = (date.today() - timedelta(days=1)).isoformat()

PATH_BRONZE_PURCHASE = f"{BASE}/bronze/purchase"
PATH_BRONZE_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA    = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE = f"{BASE}/silver/purchase_daily"
PATH_SILVER_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA    = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD_GMV        = f"{BASE}/gold/gmv_daily_by_subsidiary"

In [5]:
df_purchase_bronze_seed = spark.createDataFrame(
    [
        ("2023-01-20 22:00:00","2023-01-20",55, 15947, 5,      "2023-01-20","2023-01-20",852852),
        ("2023-01-26 00:01:00","2023-01-26",56, 369798,746520, "2023-01-25",None,        963963),
        ("2023-02-05 10:00:00","2023-02-05",55, 160001,5,      "2023-01-20","2023-01-20",852852),
        ("2023-02-26 03:00:00","2023-02-26",69, 160001,18,     "2023-02-26","2023-02-28",96967),
        ("2023-07-15 09:00:00","2023-07-15",55, 160001,5,      "2023-01-20","2023-03-01",852852),
    ],
    """
    transaction_datetime string,
    transaction_date string,
    purchase_id long,
    buyer_id long,
    prod_item_id long,
    order_date string,
    release_date string,
    producer_id long
    """
).withColumn("transaction_datetime", F.to_timestamp("transaction_datetime")) \
 .withColumn("transaction_date", F.to_date("transaction_date")) \
 .withColumn("order_date", F.to_date("order_date")) \
 .withColumn("release_date", F.to_date("release_date"))

df_purchase_bronze_seed.write.format("delta").mode("append").partitionBy("transaction_date").save(PATH_BRONZE_PURCHASE)